In [3]:
!pip install deep-translator

Defaulting to user installation because normal site-packages is not writeable


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\Ayan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
pip show beautifulsoup4

Name: beautifulsoup4
Version: 4.15.0
Summary: Screen-scraping library
Home-page: 
Author: 
Author-email: Leonard Richardson <leonardr@segfault.org>
License: MIT License
Location: C:\Users\Ayan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages
Requires: soupsieve, typing-extensions
Required-by: deep-translator
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install --upgrade beautifulsoup4 deep-translator

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: C:\Users\Ayan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

def translate_to_dholuo(text, retries=3):
    url = "https://translate.googleapis.com/translate_a/single"
    params = {
        "client": "gtx",
        "sl": "en",
        "tl": "luo",
        "dt": "t",
        "q": text
    }
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()
            return "".join([chunk[0] for chunk in data[0]])
        except Exception:
            time.sleep(1)
    return ""

import time

def translate_row(i, text):
    return i, translate_to_dholuo(text)

# Load the full file
df_keep = pd.read_csv("Combined_PSA_Raw.csv")

results = {}
texts = list(enumerate(df_keep['English']))

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(translate_row, i, text) for i, text in texts]
    for count, future in enumerate(as_completed(futures)):
        i, translated = future.result()
        results[i] = translated
        if (count + 1) % 500 == 0:
            print(f"{count+1}/{len(texts)} done")

df_keep['Dholuo'] = [results[i] for i in range(len(df_keep))]
df_keep.to_csv("PSA_Translated.csv", index=False, encoding='utf-8-sig')

500/9551 done
1000/9551 done
1500/9551 done
2000/9551 done
2500/9551 done
3000/9551 done
3500/9551 done
4000/9551 done
4500/9551 done
5000/9551 done
5500/9551 done
6000/9551 done
6500/9551 done
7000/9551 done
7500/9551 done
8000/9551 done
8500/9551 done
9000/9551 done
9500/9551 done
